# md_combat — Python vs R comparison

This notebook compares **md_combat** (Python) to **R Bioconductor `sva`** on the same bundled datasets.

**Warning — long runtime:** the **full airway** section runs **ComBatSeq** (standard, per-gene Nelder–Mead) on **~64k genes**. That step often takes **tens of minutes to hours**. `ComBatSeqFast` is much faster. Run when you can leave the machine working.

**Sections**
1. Setup (paths, helpers, figures directory)
2. ComBat (full bladderbatch) — Python vs R + figure
3. ComBatSeq **full airway** — timed standard + timed fast vs R benchmark reference + figures
4. Optional: save Python full outputs to `docs/cache/` (gitignored)
5. Optional quick **20K** smoke (subset) — faster parity check
6. Summary table

**R benchmark reference** Parquet files live in `tests/expected/` (generate with `Rscript scripts/generate_expected.R`). Python env: `uv sync --extra dev`.


## 1. Setup


In [8]:
import logging
import shutil
import subprocess
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from md_combat.combat import ComBat
from md_combat.combat_seq import ComBatSeq, ComBatSeqFast
from md_combat.datasets import load_airway, load_bladderbatch

# Suppress INFO from ComBat (logging)
logging.getLogger("md_combat.combat").setLevel(logging.WARNING)

REPO_ROOT = Path("..").resolve()
EXPECTED_DIR = REPO_ROOT / "tests" / "expected"
FIGURES_DIR = REPO_ROOT / "docs" / "figures" / "parity"
CACHE_DIR = REPO_ROOT / "docs" / "cache"
_RSCRIPT_OPTS = ["--no-save", "--no-restore", "--quiet"]

# Set True to run the optional 20K smoke section at the end
RUN_20K_SMOKE = False


def _find_rscript():
    for name in [
        "Rscript",
        "Rscript4.5-arm64",
        "Rscript4.5",
        "Rscript4.4",
        "Rscript4.3",
        "Rscript4.2",
    ]:
        path = shutil.which(name)
        if path:
            return path
    for base in (Path("/usr/local/bin"), Path("/opt/homebrew/bin")):
        if base.is_dir():
            for p in sorted(base.glob("Rscript*")):
                if p.is_file() and "Rscript" in p.name:
                    return str(p)
    return None


def run_r(script: str, timeout: int = 300) -> str:
    rscript = _find_rscript()
    if rscript is None:
        raise RuntimeError("Rscript not found on PATH")
    result = subprocess.run(
        [rscript, *_RSCRIPT_OPTS, "-e", script],
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    if result.returncode != 0:
        raise RuntimeError(f"R failed:\n{result.stderr}")
    return result.stdout


def pearson_r(a: np.ndarray, b: np.ndarray) -> float:
    x = a.ravel().astype(float)
    y = b.ravel().astype(float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 2:
        return float("nan")
    return float(np.corrcoef(x[m], y[m])[0, 1])


def corr_and_r2(a: np.ndarray, b: np.ndarray) -> tuple[float, float]:
    r = pearson_r(a, b)
    return r, r * r


def plot_11_hexbin(
    x: np.ndarray,
    y: np.ndarray,
    xlabel: str,
    ylabel: str,
    title: str,
    outfile: Path,
    max_points: int = 120_000,
    seed: int = 0,
) -> tuple[float, float]:
    rng = np.random.default_rng(seed)
    xf = x.ravel().astype(float)
    yf = y.ravel().astype(float)
    m = np.isfinite(xf) & np.isfinite(yf)
    xf, yf = xf[m], yf[m]
    n = xf.size
    if n > max_points:
        idx = rng.choice(n, size=max_points, replace=False)
        xf, yf = xf[idx], yf[idx]
    r = float(np.corrcoef(xf, yf)[0, 1])
    r2 = r * r
    fig, ax = plt.subplots(figsize=(7, 7))
    hb = ax.hexbin(xf, yf, gridsize=110, mincnt=1, linewidths=0)
    plt.colorbar(hb, ax=ax, label="count")
    lo = min(float(np.min(xf)), float(np.min(yf)))
    hi = max(float(np.max(xf)), float(np.max(yf)))
    ax.plot([lo, hi], [lo, hi], color="C3", ls="--", lw=0.45, label="1:1")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f"{title}\nPearson r = {r:.6f}   R² = {r2:.6f}")
    ax.legend(loc="upper left")
    outfile.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(outfile, dpi=300, bbox_inches="tight", pad_inches=0.02)
    plt.close(fig)
    return r, r2


RSCRIPT = _find_rscript()
print(f"Rscript: {RSCRIPT}")
if RSCRIPT:
    print(run_r("cat(R.version$version.string)").strip())
print(f"Figures -> {FIGURES_DIR}")


Rscript: /usr/local/bin/Rscript
R version 4.5.3 (2026-03-11)
Figures -> /Users/giuseppeinfusini/wd/md-repos/md_combat/docs/figures/parity


## 2. ComBat — Python vs R (full bladderbatch)

**Dataset:** 22 283 genes × 57 samples (5 batches).  
**Values:** microarray **log₂-normalised expression** (`exprs` of the Bioconductor `ExpressionSet`); ComBat returns **batch-corrected** values on the **same log₂ scale** (not raw intensities).  
**Expected:** max |difference| $\ll 10^{-3}$ vs R benchmark reference.


In [9]:
expr_df, batch_bb, _ = load_bladderbatch()
print(f"bladderbatch: {expr_df.shape[0]} genes × {expr_df.shape[1]} samples")


bladderbatch: 22283 genes × 57 samples


In [10]:
py_combat = ComBat().fit_transform(expr_df, batch_bb)
print(f"Python ComBat: {py_combat.shape}")


Python ComBat: (22283, 57)


In [11]:
r_combat = pd.read_parquet(EXPECTED_DIR / "bladderbatch_combat.parquet")
print(f"R sva::ComBat (benchmark ref): {r_combat.shape}")


R sva::ComBat (benchmark ref): (22283, 57)


In [12]:
max_diff_combat = np.abs(py_combat.values - r_combat.values).max()
r_combat_p, r2_combat_p = corr_and_r2(py_combat.values, r_combat.values)
print(f"Max |diff| : {max_diff_combat:.2e}")
print(f"Pearson r  : {r_combat_p:.10f}")
print(f"R² (= r²)  : {r2_combat_p:.10f}")

plot_11_hexbin(
    r_combat.values,
    py_combat.values,
    "R: batch-corrected log₂ expression",
    "Python: batch-corrected log₂ expression",
    "ComBat — bladderbatch (each point = one gene × sample)",
    FIGURES_DIR / "combat_bladderbatch_11.png",
)
print(f"Saved {FIGURES_DIR / 'combat_bladderbatch_11.png'}")


Max |diff| : 6.22e-15
Pearson r  : 1.0000000000
R² (= r²)  : 1.0000000000
Saved /Users/giuseppeinfusini/wd/md-repos/md_combat/docs/figures/parity/combat_bladderbatch_11.png


## 3. ComBatSeq — full airway (timed standard + timed fast)

**Prerequisite:** `tests/expected/airway_full_combat_seq.parquet` from:

`Rscript scripts/generate_expected.R`

**Python:** `ComBatSeq` (statsmodels NM per gene) then `ComBatSeqFast` (vectorised Newton–Raphson). Both compared to the same R benchmark reference matrix.


In [13]:
counts_full, batch_aw, group_aw = load_airway()
print(f"airway full: {counts_full.shape[0]} genes × {counts_full.shape[1]} samples")

full_r_path = EXPECTED_DIR / "airway_full_combat_seq.parquet"
if not full_r_path.is_file():
    raise FileNotFoundError(
        f"Missing {full_r_path}. Run from repo root: Rscript scripts/generate_expected.R"
    )
r_seq_full = pd.read_parquet(full_r_path)
print(f"R ComBat_seq full (benchmark ref): {r_seq_full.shape}")
assert r_seq_full.shape == counts_full.shape


airway full: 64102 genes × 8 samples
R ComBat_seq full (benchmark ref): (64102, 8)


In [14]:
print("Running ComBatSeq (standard) on full airway — this may take a long time...")
t0 = time.perf_counter()
py_seq_std_full = ComBatSeq().fit_transform(counts_full, batch_aw, group=group_aw)
TIME_SEQ_STD_FULL = time.perf_counter() - t0
print(f"Done. Wall time: {TIME_SEQ_STD_FULL:.1f} s ({TIME_SEQ_STD_FULL/60:.2f} min)")
print(py_seq_std_full.shape)


Running ComBatSeq (standard) on full airway — this may take a long time...
Done. Wall time: 395.6 s (6.59 min)
(64102, 8)


In [15]:
print("Running ComBatSeqFast on full airway...")
t0 = time.perf_counter()
py_seq_fast_full = ComBatSeqFast().fit_transform(counts_full, batch_aw, group=group_aw)
TIME_SEQ_FAST_FULL = time.perf_counter() - t0
print(f"Done. Wall time: {TIME_SEQ_FAST_FULL:.1f} s ({TIME_SEQ_FAST_FULL/60:.2f} min)")
print(py_seq_fast_full.shape)
if TIME_SEQ_FAST_FULL > 0:
    print(f"Speedup (std / fast): {TIME_SEQ_STD_FULL / TIME_SEQ_FAST_FULL:.1f}×")


Running ComBatSeqFast on full airway...
Done. Wall time: 1.9 s (0.03 min)
(64102, 8)
Speedup (std / fast): 205.8×


In [16]:
r_std_full, r2_std_full = corr_and_r2(py_seq_std_full.values, r_seq_full.values)
r_fast_full, r2_fast_full = corr_and_r2(py_seq_fast_full.values, r_seq_full.values)
r_fvs_full, r2_fvs_full = corr_and_r2(py_seq_fast_full.values, py_seq_std_full.values)

print("vs R benchmark reference (full airway):")
print(f"  ComBatSeq (standard):  Pearson r = {r_std_full:.6f}   R² = {r2_std_full:.6f}")
print(f"  ComBatSeqFast:         Pearson r = {r_fast_full:.6f}   R² = {r2_fast_full:.6f}")
print("Python standard vs Python fast (full):")
print(f"  Pearson r = {r_fvs_full:.6f}   R² = {r2_fvs_full:.6f}")


vs R benchmark reference (full airway):
  ComBatSeq (standard):  Pearson r = 0.999995   R² = 0.999990
  ComBatSeqFast:         Pearson r = 0.996520   R² = 0.993053
Python standard vs Python fast (full):
  Pearson r = 0.996521   R² = 0.993054


In [17]:
plot_11_hexbin(
    r_seq_full.values,
    py_seq_std_full.values,
    "R sva::ComBat_seq",
    "Python ComBatSeq",
    "ComBatSeq — full airway (standard Python)",
    FIGURES_DIR / "combatseq_airway_full_std_vs_r.png",
)
plot_11_hexbin(
    r_seq_full.values,
    py_seq_fast_full.values,
    "R sva::ComBat_seq",
    "Python ComBatSeqFast",
    "ComBatSeq — full airway (fast Python)",
    FIGURES_DIR / "combatseq_airway_full_fast_vs_r.png",
)
plot_11_hexbin(
    py_seq_std_full.values,
    py_seq_fast_full.values,
    "Python ComBatSeq",
    "Python ComBatSeqFast",
    "ComBatSeq vs ComBatSeqFast — full airway",
    FIGURES_DIR / "combatseq_airway_full_std_vs_fast.png",
)
print("Saved full-airway parity figures to", FIGURES_DIR)


Saved full-airway parity figures to /Users/giuseppeinfusini/wd/md-repos/md_combat/docs/figures/parity


## 4. Optional — cache Python full matrices

Uncomment to write parquet under `docs/cache/` (gitignored) after a successful full run.


In [ ]:
# CACHE_DIR.mkdir(parents=True, exist_ok=True)
# py_seq_std_full.to_parquet(CACHE_DIR / "airway_full_combatseq_std.parquet")
# py_seq_fast_full.to_parquet(CACHE_DIR / "airway_full_combatseq_fast.parquet")
# print("Wrote cache parquets")


## 5. Optional — 20K subset smoke test

Set `RUN_20K_SMOKE = True` in the setup cell to run a faster ComBatSeq check (~few minutes for standard).


In [ ]:
if RUN_20K_SMOKE:
    gene_idx = pd.read_parquet(EXPECTED_DIR / "airway_20k_gene_ids.parquet")["gene_idx"].tolist()
    counts_20k = counts_full.iloc[gene_idx]
    t0 = time.perf_counter()
    py_seq_std = ComBatSeq().fit_transform(counts_20k, batch_aw, group=group_aw)
    t_std = time.perf_counter() - t0
    t0 = time.perf_counter()
    py_seq_fast = ComBatSeqFast().fit_transform(counts_20k, batch_aw, group=group_aw)
    t_fast = time.perf_counter() - t0
    r_seq_20k = pd.read_parquet(EXPECTED_DIR / "airway_20k_combat_seq.parquet")
    r20_std, _ = corr_and_r2(py_seq_std.values, r_seq_20k.values)
    r20_fast, _ = corr_and_r2(py_seq_fast.values, r_seq_20k.values)
    r20_fvs, _ = corr_and_r2(py_seq_fast.values, py_seq_std.values)
    print(f"20K std time {t_std:.1f}s, fast {t_fast:.1f}s, speedup {t_std/t_fast:.1f}x")
    print(f"20K vs R: std r={r20_std:.4f}, fast r={r20_fast:.4f}, fast vs std r={r20_fvs:.4f}")
else:
    py_seq_std = py_seq_fast = r_seq_20k = None
    t_std = t_fast = float("nan")
    r20_std = r20_fast = r20_fvs = float("nan")
    print("Skipped 20K smoke (RUN_20K_SMOKE=False)")


## 6. Summary table


In [ ]:
rows = [
    {
        "comparison": "ComBat Python vs R",
        "dataset": "bladderbatch (full)",
        "metric": "max |diff|",
        "value": f"{max_diff_combat:.2e}",
        "time_s": "",
        "threshold": "< 1e-3",
        "pass": max_diff_combat < 1e-3,
    },
    {
        "comparison": "ComBatSeq (std) vs R",
        "dataset": "airway (full)",
        "metric": "Pearson r / R²",
        "value": f"{r_std_full:.5f} / {r2_std_full:.5f}",
        "time_s": f"{TIME_SEQ_STD_FULL:.1f}",
        "threshold": "> 0.95",
        "pass": r_std_full > 0.95,
    },
    {
        "comparison": "ComBatSeqFast vs R",
        "dataset": "airway (full)",
        "metric": "Pearson r / R²",
        "value": f"{r_fast_full:.5f} / {r2_fast_full:.5f}",
        "time_s": f"{TIME_SEQ_FAST_FULL:.1f}",
        "threshold": "> 0.95",
        "pass": r_fast_full > 0.95,
    },
    {
        "comparison": "ComBatSeqFast vs ComBatSeq",
        "dataset": "airway (full)",
        "metric": "Pearson r / R²",
        "value": f"{r_fvs_full:.5f} / {r2_fvs_full:.5f}",
        "time_s": "",
        "threshold": "> 0.99",
        "pass": r_fvs_full > 0.99,
    },
]
if RUN_20K_SMOKE:
    rows.append(
        {
            "comparison": "20K smoke: fast vs std r",
            "dataset": "airway 20K",
            "metric": "Pearson r",
            "value": f"{r20_fvs:.4f}",
            "time_s": f"std {t_std:.0f}s / fast {t_fast:.0f}s",
            "threshold": "> 0.99",
            "pass": r20_fvs > 0.99,
        }
    )

summary = pd.DataFrame(rows)
summary["status"] = summary["pass"].map({True: "PASS", False: "FAIL"})
display(summary.drop(columns="pass"))
overall = summary["pass"].all()
print(f"\nOverall: {'ALL PASS' if overall else 'SOME FAILURES'}")
